## CORA water level datasets are referenced to mean sea level (MSL). This notebook demonstrates how to convert the data to another tidal datum (Mean Higher High Water, MHHW) using NOAA's Tidal Analysis Datum Calculator (TADC).

Access all the data in the S3 bucket here: https://noaa-nos-cora-pds.s3.amazonaws.com/index.html

*Parts of the code in this notebook were guided by **Signell, R. (2024). Reproducible Notebooks/hurricane-ike-water-levels. https://github.com/reproducible-notebooks/hurricane-ike-water-levels/tree/master***

_ _ 

### Input a location of interest in the format (lat, lon) ###

In [ ]:
loc = (36.62, -121.89)  # Near Monterey, CA

### Import packages ###

In [ ]:
import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pprint import pprint
import tadc  # NOAA's Tidal Analysis Datum Calculator #
import zarr

### Initialize and list the available CORA datasets ###
Description of datasets: <br>
**CORA-{domain}-fort.63-{chunking}**: Hourly water levels <br>
**CORA-{domain}-swan_TPS.63-{chunking}**: Hourly spectral peak wave periods <br>
**CORA-{domain}-swan_DIR.63-{chunking}**: Hourly mean wave directions <br>
**CORA-{domain}-swan_HS.63-{chunking}**: Hourly significant wave heights <br>
**CORA-{domain}-Grid-{chunking}**: Hourly water levels interpolated from model nodes to a uniform 500 meter resolution grid <br>

Where: <br>
**{domain}** represents the domain. Options are **GEC** or **PAC**. <br>
**{chunking}** represents the chunking strategy, which affects data loading speed. Use **timeseries** if you are interested in long timeseries at a few points, use **map** if you are interested in many points for a few time periods (up to a few days).

In [ ]:
if int(zarr.__version__[0]) > 2:  ## There are some issues with zarr 2 vs. zarr 3, so two catalogs have been created depdning on the zarr version you are using.
    catalog_file = 'CORA_intake_zarr3.yml'
else:
    catalog_file = 'CORA_intake.yml'

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/"+catalog_file,storage_options={'anon':True})
list(catalog)

### Initialize CORA-PAC water levels (see full dataset [here](https://noaa-nos-cora-pds.s3.amazonaws.com/index.html#cora_pac/native_grid/water_levels/)). ###
#### This dataset contains hourly water levels from 1979-2025 in the PAC domain. The dataset is multiple TB in size and cannot be loaded into memory on a standard computer. We use the xarray package to "lazily" load the data - initializing it and only loading the pieces that we need later. Note that we use the **timeseries** chunking here, since we are interested in a long timeseries at a single point.

In [ ]:
ds = catalog["CORA-PAC-fort.63-timeseries"].to_dask()
ds

### Find the closest node to the location of interest (using the haversine formula) and load the time series for the time period 1983-2001

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    R = 6371 
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = R * c
    return km 

d = haversine(loc[0],loc[1],ds.y,ds.x)
closest_node = int(np.argmin(d.data))
ts = ds['zeta'].sel(node=closest_node,time=slice("1983-1-1","2001-12-31"))
ts

### Compute tidal datums from the data using the Tidal Analysis Datum Calculator

In [ ]:
data = pd.DataFrame({'time':ts['time'].values,'val':ts.values})

tadc_out = tadc.run(data=data, Subordinate_Lat=loc[0], Subordinate_Lon=loc[1], Units='millimeters')
datums = tadc_out.datums
pprint(datums)

### Convert the data to MHHW and plot it ###
#### We will convert the CORA data to units of meters for the plot

In [ ]:
zeta_mhhw = data['val'] - datums['MHHW']

fig,ax = plt.subplots(1,figsize=(8,4))
ax.plot(data['time'],zeta_mhhw/1000)
ax.set_ylabel('Water level (m, MHHW)')
fig.autofmt_xdate()